# OptiForge · Part 1: causal ECG conditioning + PSO tuning

**What this notebook does:** loads real MIT-BIH ECG, damages it with real NSTDB noise, cleans it with a **causal** (real-time-safe) filter, detects R-peaks with a streaming-style Pan-Tompkins, and lets **PSO** tune the filter cut-offs and detector threshold while protecting QRS amplitude and ST level. Everything is scored on **held-out records**.

**How to run:** Runtime → Run all (about 5-8 min). Set `MODE = "quick"` in cell 1 for a 1-minute smoke test.
**Coming in Part 2:** fuzzy classifier + PSO, VF/asystole paths, 3-second alert loop.

## 1 · Setup and parameter configuration

In [ ]:
!pip -q install wfdb
import os, time, json, zipfile, glob, tracemalloc, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import butter, iirnotch, tf2sos, sosfilt, sosfilt_zi, correlate
try:
    from IPython.display import display
except Exception:
    display = print
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# ============================ PARAMETER CONFIGURATION ============================
MODE = "full"                      # "quick" = ~1 min smoke test | "full" = complete run
FS, DUR_S, WARM_S = 360, 60, 2.0   # sampling rate (Hz) | seconds used per record | detector learning period (s)
TRAIN_RECS = ["100", "101", "103", "106"]     # PSO sees ONLY these MIT-BIH records
TEST_RECS  = ["105", "119", "200", "201"]     # held out: never used for tuning
TRAIN_NOISE, TRAIN_SNR = ["bw", "em", "ma"], [0, -6]           # scenarios PSO trains on (all 3 noise types)
TEST_NOISE,  TEST_SNR  = ["bw", "em", "ma"], [6, 0, -6, -12]   # scenarios in the final report
N_SEG = 2                          # random noise segments per test scenario
DEFAULT = dict(hp=0.5, lp=40.0, thr=0.25)     # hand-picked baseline: high-pass Hz, low-pass Hz, detector threshold
LO, HI = np.array([0.05, 25.0, 0.05]), np.array([0.67, 100.0, 0.80])   # PSO bounds = clinical limits: hp <= 0.67 Hz (AHA ST-safe), lp >= 25 Hz (keeps QRS energy)
PSO = dict(n=20, iters=30, w0=0.9, w1=0.4, c1=1.5, c2=1.5)             # particles, iterations, inertia 0.9 -> 0.4, c1, c2
SEEDS = [0, 1, 2]                  # independent PSO runs (stability evidence)
# fitness = W_F1*(1-F1) + W_JIT*jitter + W_AMP*amp_err + W_ST*st_err - W_BW*drift_snr_gain - W_SNR*snr_gain  (+2 per violated hard limit)
W = dict(F1=1.0, JIT=0.2, AMP=1.0, ST=0.3, BW=0.03, SNR=0.01)
AMP_MAX, ST_MAX = 0.10, 0.05       # hard limits: QRS amplitude error (fraction) and ST-level error (mV)
NOTCH_HZ = 50                      # mains frequency in India
if MODE == "quick":
    DUR_S, TRAIN_RECS, TEST_RECS, SEEDS = 30, TRAIN_RECS[:2], TEST_RECS[:2], SEEDS[:1]
    PSO["iters"], N_SEG = 8, 1
os.makedirs("results", exist_ok=True)
print(f"Mode: {MODE} | train {TRAIN_RECS} | test {TEST_RECS} | PSO seeds {SEEDS}")

## 2 · Data: real PhysioNet records (automatic synthetic fallback if offline)

In [ ]:
BEATS = set("NLRBAaJSVrFejnE/fQ?")    # annotation symbols that mean "this is a heartbeat"
NOISE_S = max(300, 3 * DUR_S)         # seconds of each noise record kept (random windows are cut from it)

def synth_ecg(seconds, seed):         # FALLBACK ONLY: sum-of-Gaussians ECG plus exact beat positions
    rng = np.random.default_rng(seed); n = int(seconds * FS); t = np.arange(n) / FS
    x, beats, pos, hr = np.zeros(n), [], rng.uniform(0.4, 0.9), rng.uniform(60, 95)
    while pos < seconds - 0.6:
        beats.append(int(pos * FS))
        for a, mu, s in [(0.12, -0.20, 0.025), (-0.10, -0.04, 0.010), (1.10, 0, 0.012), (-0.25, 0.035, 0.012), (0.30, 0.25, 0.045)]:
            x += a * rng.uniform(0.9, 1.1) * np.exp(-0.5 * ((t - pos - mu) / s) ** 2)
        pos += 60 / hr * rng.uniform(0.92, 1.08)
    return x + 0.01 * rng.standard_normal(n), np.array(beats)

def synth_noise(kind, seed):          # FALLBACK ONLY
    rng = np.random.default_rng(seed); w = rng.standard_normal(NOISE_S * FS)
    typ, edge = {"bw": ("lowpass", 0.5), "em": ("bandpass", [20, 120]), "ma": ("bandpass", [5, 60])}[kind]
    y = sosfilt(butter(2, edge, typ, fs=FS, output="sos"), w)
    if kind == "ma":
        slow = sosfilt(butter(2, 0.2, "lowpass", fs=FS, output="sos"), rng.standard_normal(len(w)))
        y = y * (0.3 + np.abs(slow) / slow.std())
    return y

def load_real(rec):
    import wfdb
    n = DUR_S * FS
    r, a = wfdb.rdrecord(rec, pn_dir="mitdb", sampto=n), wfdb.rdann(rec, "atr", pn_dir="mitdb", sampto=n)
    keep = np.array([s in BEATS for s in a.symbol], dtype=bool)
    truth = np.asarray(a.sample)[keep]
    return np.nan_to_num(r.p_signal[:, 0]), truth[truth < n]

def load_noise(kind):
    import wfdb
    return np.nan_to_num(wfdb.rdrecord(kind, pn_dir="nstdb", sampto=NOISE_S * FS).p_signal[:, 0])

RECS = TRAIN_RECS + TEST_RECS
try:
    DATA  = {r: load_real(r) for r in RECS}
    NOISE = {k: load_noise(k) for k in ["bw", "em", "ma"]}
    SYNTHETIC = False
except Exception as e:
    print(f"!! PhysioNet not reachable ({type(e).__name__}). Using SYNTHETIC data. Do NOT quote these numbers as real-data results.")
    DATA  = {r: synth_ecg(DUR_S, int(r)) for r in RECS}
    NOISE = {k: synth_noise(k, i) for i, k in enumerate(["bw", "em", "ma"])}
    SYNTHETIC = True

def add_noise(clean, noise, snr_db):  # scale `noise` so the mix has exactly `snr_db` dB signal-to-noise
    c0, n0 = clean - clean.mean(), noise - noise.mean()
    return clean + np.sqrt(np.mean(c0 ** 2) / (np.mean(n0 ** 2) * 10 ** (snr_db / 10) + 1e-12)) * n0

def make_scenarios(recs, kinds, snrs, n_seg, seed):
    rng, out = np.random.default_rng(seed), []
    for rec in recs:
        ref, truth = DATA[rec]
        for kind in kinds:
            for snr in snrs:
                for _ in range(n_seg):
                    o = int(rng.integers(0, len(NOISE[kind]) - len(ref)))
                    out.append(dict(rec=rec, noise=kind, snr=snr, ref=ref, truth=truth,
                                    noisy=add_noise(ref, NOISE[kind][o:o + len(ref)], snr)))
    return out

print("Source:", "SYNTHETIC fallback" if SYNTHETIC else "real MIT-BIH + NSTDB")
print(pd.DataFrame({r: dict(seconds=len(s) / FS, beats=len(t)) for r, (s, t) in DATA.items()}).T.astype(int).to_string())

## 3 · Core: causal filter, streaming-style QRS detector, metrics

In [ ]:
SOS_BP = butter(2, [5, 15], "bandpass", fs=FS, output="sos")      # Pan-Tompkins front-end band-pass

def make_sos(hp, lp):        # high-pass (drift) -> low-pass (muscle) -> 50 Hz notch, as ONE small filter cascade
    b, a = iirnotch(NOTCH_HZ, 30, fs=FS)
    return np.vstack([butter(2, hp, "highpass", fs=FS, output="sos"),
                      butter(4, lp, "lowpass", fs=FS, output="sos"), tf2sos(b, a)])

def apply(sos, x):           # CAUSAL: output[n] depends only on x[:n+1]
    return sosfilt(sos, x, zi=sosfilt_zi(sos) * x[0])[0]

class Stream:                # the same filter fed chunk by chunk, exactly how a live monitor runs it
    def __init__(self, sos): self.sos, self.zi = sos, None
    def __call__(self, chunk):
        if self.zi is None: self.zi = sosfilt_zi(self.sos) * chunk[0]
        y, self.zi = sosfilt(self.sos, chunk, zi=self.zi)
        return y

def group_delay(sos, ref):   # how many samples the filter delays the signal (measured, not assumed)
    y = apply(sos, ref); c = correlate(y - y.mean(), ref - ref.mean(), "full", "fft")
    lag = np.arange(-len(ref) + 1, len(ref)); ok = (lag >= 0) & (lag <= FS // 2)
    return int(lag[ok][np.argmax(c[ok])])

def detect(x, thr, delay=0):
    # causal Pan-Tompkins: band-pass -> derivative -> square -> 150 ms trailing average -> adaptive threshold
    w0, w = int(WARM_S * FS), int(0.15 * FS)
    if len(x) <= w0 + w: return np.array([], dtype=int)
    bp = sosfilt(SOS_BP, x, zi=sosfilt_zi(SOS_BP) * x[0])[0]
    cs = np.cumsum(np.diff(bp, prepend=bp[0]) ** 2)
    m = (cs - np.r_[np.zeros(w), cs[:-w]]) / w
    spk, npk, last, rr, pk = 0.5 * m[:w0].max(), float(np.median(m[:w0])), -10 ** 9, 0.8 * FS, []
    cand = np.flatnonzero((m[1:-1] > m[:-2]) & (m[1:-1] >= m[2:])) + 1
    for c in cand[cand >= w0]:
        gap, v = c - last, m[c]
        if gap < 0.2 * FS: continue                                   # 200 ms refractory period
        if gap > 3 * FS: spk *= 0.9                                   # decay so a noise burst can never lock the detector
        th = npk + thr * (spk - npk)
        if v > th or (gap > 1.66 * rr and v > 0.5 * th):              # normal detection | search-back for a missed beat
            spk = 0.125 * v + 0.875 * spk
            if last > 0: rr = float(np.clip(0.875 * rr + 0.125 * gap, 0.3 * FS, 2 * FS))
            pk.append(c); last = c
        else:
            npk = 0.125 * v + 0.875 * npk
    back, out = int(0.25 * FS), []
    for c in pk:                                                      # snap to the true R-peak, looking backwards only
        seg = x[c - back:c + 1]; out.append(c - back + int(np.argmax(np.abs(seg - np.median(seg)))))
    return np.unique(np.maximum(np.array(out, dtype=int) - delay, 0))

def score(pk, truth, tol_ms=75):
    w0 = int(WARM_S * FS); truth, pk = truth[truth >= w0], pk[pk >= w0]
    tol, used, errs = int(tol_ms * FS / 1000), set(), []
    for p in pk:
        if len(truth) == 0: break
        j = int(np.argmin(np.abs(truth - p)))
        if abs(truth[j] - p) <= tol and j not in used: used.add(j); errs.append(p - truth[j])
    tp = len(used); fp, fn = len(pk) - tp, len(truth) - tp
    pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
    return dict(F1=2 * pr * rc / (pr + rc) if pr + rc else 0.0, prec=pr, recall=rc,
                jit=float(np.std(errs)) * 1000 / FS if len(errs) > 1 else 0.0)      # jit = R-peak timing jitter (ms)

def snr_db(ref, x, d=0):
    if d: ref, x = ref[:-d], x[d:]
    r0, x0 = ref - ref.mean(), x - x.mean()
    return 10 * np.log10((np.sum(r0 ** 2) + 1e-12) / (np.sum((x0 - r0) ** 2) + 1e-12))

A, I0, I1, S0, S1 = int(.05 * FS), int(.08 * FS), int(.05 * FS), int(.09 * FS), int(.11 * FS)   # QRS half-width, PR-baseline and ST windows
def _st(z, k): return z[k + S0:k + S1].mean() - z[k - I0:k - I1].mean()                            # ST level relative to the PR baseline

def shape_err(ref, truth, sos, d, x=None):   # QRS-amplitude error (fraction) and ST-level error (mV) vs the reference
    y = apply(sos, ref if x is None else x)[d:]; r = ref[:len(y)]                                  # x=None -> filter the CLEAN signal
    ea, es = [], []
    for k in truth[(truth >= int(WARM_S * FS)) & (truth < len(y) - S1)]:
        pa = np.ptp(r[k - A:k + A + 1]); ea.append(abs(np.ptp(y[k - A:k + A + 1]) - pa) / max(pa, 1e-3))
        es.append(abs(_st(y, k) - _st(r, k)))
    return (float(np.mean(ea)), float(np.mean(es))) if ea else (0.0, 0.0)

def run_scenario(s, sos, thr, d):
    f = apply(sos, s["noisy"]); m = score(detect(f, thr, d), s["truth"])
    m["snr_gain"] = snr_db(s["ref"], f, d) - snr_db(s["ref"], s["noisy"])
    # SNR gain on baseline-wander scenarios only: rewards drift removal, so PSO cannot simply switch the high-pass off
    m["bw_gain"] = m["snr_gain"] if s["noise"] == "bw" else np.nan
    return m

def terms(p, sc, refs):              # the fitness, with every term returned separately
    hp, lp, thr = p; sos = make_sos(hp, lp); d = group_delay(sos, refs[0][0])
    L = [run_scenario(s, sos, thr, d) for s in sc]
    vals = lambda k: [m[k] for m in L if np.isfinite(m[k])]
    ms = {k: float(np.mean(vals(k))) if vals(k) else 0.0 for k in L[0]}
    amp, st = np.mean([shape_err(r, t, sos, d) for r, t in refs], axis=0)
    cost = (W["F1"] * (1 - ms["F1"]) + W["JIT"] * min(ms["jit"] / 20, 2) + W["AMP"] * amp + W["ST"] * min(st / 0.1, 3)
            - W["BW"] * ms["bw_gain"] - W["SNR"] * ms["snr_gain"] + 2.0 * (amp > AMP_MAX) + 2.0 * (st > ST_MAX))
    return dict(cost=float(cost), F1=ms["F1"], jitter_ms=ms["jit"], amp_err=float(amp), st_err_mV=float(st),
                bw_gain_dB=ms["bw_gain"], snr_gain=ms["snr_gain"])

## 4 · Self-checks (every line must say PASS; a NOTE is a finding, not an error)

In [ ]:
def check(name, ok, extra="", strict=True): print("PASS" if ok else ("FAIL" if strict else "NOTE"), "|", name, extra)

ref0, tr0 = DATA[TRAIN_RECS[0]]; sos0 = make_sos(DEFAULT["hp"], DEFAULT["lp"]); d0 = group_delay(sos0, ref0)
noisy0 = add_noise(ref0, NOISE["ma"][:len(ref0)], 0)
cut = int(len(ref0) * 0.55); x2 = noisy0.copy()
x2[int(len(ref0) * 0.6):] += 5 * np.random.default_rng(1).standard_normal(len(ref0) - int(len(ref0) * 0.6))   # damage only the FUTURE
p1, p2 = detect(apply(sos0, noisy0), DEFAULT["thr"], d0), detect(apply(sos0, x2), DEFAULT["thr"], d0)
check("filter is causal: changing the future never changes the past", np.allclose(apply(sos0, noisy0)[:cut], apply(sos0, x2)[:cut]))
check("detector is causal: same beats before the change", np.array_equal(p1[p1 < cut - FS], p2[p2 < cut - FS]))
S = Stream(sos0)
check("streaming in 100 ms chunks == whole-signal filtering",
      np.allclose(np.concatenate([S(c) for c in np.array_split(noisy0, len(noisy0) // 36)]), apply(sos0, noisy0)))
check("noise mixer hits the requested SNR", abs(snr_db(ref0, add_noise(ref0, NOISE["em"][:len(ref0)], 6)) - 6) < 0.01)
f1c = score(detect(apply(sos0, ref0), DEFAULT["thr"], d0), tr0)["F1"]
check("detector finds the beats on a clean signal", f1c >= 0.9, f"(F1 = {f1c:.3f})")
amp0, st0 = shape_err(ref0, tr0, sos0, d0)
check("default filter stays inside the QRS/ST limits", amp0 <= AMP_MAX and st0 <= ST_MAX, f"(amp err {amp0:.3f}, ST err {st0:.4f} mV)", strict=False)
tracemalloc.start(); _ = detect(apply(sos0, noisy0[:10 * FS]), DEFAULT["thr"], d0); peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
print(f"\nFilter delay: {d0} samples = {1000 * d0 / FS:.1f} ms (added to the 3 s alert budget)")
print(f"Filter cascade: {len(sos0)} sections, {2 * len(sos0)} state values | Python peak memory for a 10 s buffer: {peak / 1024:.0f} KB")

## 5 · Attempt 1: baseline with hand-picked settings

In [ ]:
TRAIN_SC = make_scenarios(TRAIN_RECS, TRAIN_NOISE, TRAIN_SNR, 1, seed=1)
TEST_SC  = make_scenarios(TEST_RECS, TEST_NOISE, TEST_SNR, N_SEG, seed=2)
TRAIN_REFS, TEST_REFS = [DATA[r] for r in TRAIN_RECS], [DATA[r] for r in TEST_RECS]
P0 = np.array([DEFAULT["hp"], DEFAULT["lp"], DEFAULT["thr"]])

def shape_row(p):
    sos = make_sos(*p[:2]); d = group_delay(sos, DATA[TRAIN_RECS[0]][0])
    a, s = np.mean([shape_err(r, t, sos, d) for r, t in TEST_REFS], axis=0)
    return dict(amp_err=a, st_err_mV=s, delay_ms=1000 * d / FS)

def evaluate(p, sc, name):
    sos = make_sos(*p[:2]); d = group_delay(sos, DATA[TRAIN_RECS[0]][0])
    return pd.DataFrame([dict(config=name, rec=s["rec"], noise=s["noise"], snr_in=s["snr"], **run_scenario(s, sos, p[2], d)) for s in sc])

base = terms(P0, TRAIN_SC, TRAIN_REFS)
base_test = evaluate(P0, TEST_SC, "1 default")
ATTEMPTS = [dict(attempt=1, note="baseline: causal filter + adaptive Pan-Tompkins, hand-picked settings",
                 train_fitness=round(base["cost"], 4), test_F1=round(base_test.F1.mean(), 3))]
print("Baseline fitness terms on TRAIN scenarios:", {k: round(v, 4) for k, v in base.items()})
print("Baseline mean F1 on HELD-OUT test scenarios:", round(base_test.F1.mean(), 3))

## 6 · Attempt 2: PSO tunes high-pass, low-pass and detector threshold

In [ ]:
def pso(fit, seed):
    # representation: particle = [hp, lp, thr]. Operators: velocity = inertia*V + c1*r1*(own best - X) + c2*r2*(swarm best - X),
    # velocity clamped to 20% of the range, position clamped to the bounds, inertia decays 0.9 -> 0.4 (explore first, then refine)
    rng = np.random.default_rng(seed); n, T, D = PSO["n"], PSO["iters"], len(LO)
    X = rng.uniform(LO, HI, (n, D)); X[0] = P0                    # particle 0 = baseline, so PSO can never end worse on train
    V, vmax = np.zeros((n, D)), 0.2 * (HI - LO)
    Pb, Pc = X.copy(), np.array([fit(x) for x in X]); g, gc = Pb[Pc.argmin()].copy(), float(Pc.min()); hist = [gc]
    for t in range(T):
        w = PSO["w0"] - (PSO["w0"] - PSO["w1"]) * t / max(1, T - 1)
        V = np.clip(w * V + PSO["c1"] * rng.random((n, D)) * (Pb - X) + PSO["c2"] * rng.random((n, D)) * (g - X), -vmax, vmax)
        X = np.clip(X + V, LO, HI)
        c = np.array([fit(x) for x in X]); b = c < Pc; Pb[b], Pc[b] = X[b], c[b]
        if Pc.min() < gc: g, gc = Pb[Pc.argmin()].copy(), float(Pc.min())
        hist.append(gc)
    return g, gc, hist

def fit(p):
    try: c = terms(p, TRAIN_SC, TRAIN_REFS)["cost"]
    except Exception: c = 10.0
    return c if np.isfinite(c) else 10.0

runs = []
for sd in SEEDS:
    t0 = time.time(); g, gc, h = pso(fit, sd)
    runs.append(dict(seed=sd, hp=g[0], lp=g[1], thr=g[2], train_cost=gc, hist=h))
    print(f"seed {sd}: hp={g[0]:.2f} Hz  lp={g[1]:.1f} Hz  thr={g[2]:.2f}  train fitness={gc:.4f}  ({time.time() - t0:.0f}s)")
best = min(runs, key=lambda r: r["train_cost"])                    # chosen by TRAIN fitness only, never by test score
PB = np.array([best["hp"], best["lp"], best["thr"]])
pd.DataFrame({f"seed{r['seed']}": r["hist"] for r in runs}).to_csv("results/pso_convergence.csv", index_label="iteration")
print("\nChosen (best train fitness):", dict(hp=round(float(PB[0]), 2), lp=round(float(PB[1]), 1), thr=round(float(PB[2]), 2)))
print("Tuned fitness terms on TRAIN:", {k: round(v, 4) for k, v in terms(PB, TRAIN_SC, TRAIN_REFS).items()})

## 7 · Results on held-out records

In [ ]:
configs = {"1 default": P0, "2 tuned thr only": np.r_[P0[:2], PB[2]],
           "3 tuned filter only": np.r_[PB[:2], P0[2]], "4 all tuned (PSO)": PB}
res = pd.concat([evaluate(p, TEST_SC, n) for n, p in configs.items()], ignore_index=True)
ablation = res.groupby("config")[["F1", "prec", "recall", "jit", "snr_gain", "bw_gain"]].mean().join(
    pd.DataFrame({n: shape_row(p) for n, p in configs.items()}).T)
print("ABLATION (which part of the tuning actually helps?), mean over all held-out scenarios"); display(ablation.round(3))

by = res[res.config.isin(["1 default", "4 all tuned (PSO)"])].pivot_table(index=["noise", "snr_in"], columns="config", values="F1")
print("F1 by noise type and input SNR"); display(by.round(3))

rows = []
for r in runs:
    p = np.array([r["hp"], r["lp"], r["thr"]])
    rows.append(dict(seed=r["seed"], hp=r["hp"], lp=r["lp"], thr=r["thr"], test_F1=evaluate(p, TEST_SC, "s").F1.mean(), **shape_row(p)))
stab = pd.DataFrame(rows)
print("STABILITY across PSO seeds"); display(stab.round(3))
print(f"test F1 = {stab.test_F1.mean():.3f} +/- {stab.test_F1.std(ddof=0):.3f} (tuned) vs {base_test.F1.mean():.3f} (default)")

ATTEMPTS.append(dict(attempt=2, note="PSO tunes hp/lp/thr with amplitude+ST penalties (hp <= 0.67 Hz) because defaults lose F1 under EMG/motion noise",
                     train_fitness=round(best["train_cost"], 4), test_F1=round(ablation.loc["4 all tuned (PSO)", "F1"], 3)))
pd.DataFrame(ATTEMPTS).to_csv("results/attempts.csv", index=False)
with open("results/attempts.md", "w") as fh:
    fh.write("# Attempts\n\n" + "\n".join(f"{a['attempt']}. {a['note']} | train fitness {a['train_fitness']} | held-out F1 {a['test_F1']}" for a in ATTEMPTS) + "\n")
display(pd.DataFrame(ATTEMPTS))

## 8 · Plots and export

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
for r in runs: ax[0].plot(r["hist"], label=f"seed {r['seed']}")
ax[0].set(title="PSO convergence (lower = better)", xlabel="iteration", ylabel="best fitness"); ax[0].legend()
for a, kind in zip(ax[1:], TEST_NOISE):
    for name, style in [("1 default", "o--"), ("4 all tuned (PSO)", "s-")]:
        q = res[(res.config == name) & (res.noise == kind)].groupby("snr_in").F1.mean()
        a.plot(q.index, q.values, style, label=name)
    a.set(title=f"F1 vs input SNR ({kind} noise)", xlabel="input SNR (dB)", ylim=(0, 1.05)); a.legend()
plt.tight_layout(); plt.savefig("results/summary.png", dpi=120); plt.show()

s = next(s for s in TEST_SC if s["noise"] == "ma" and s["snr"] == -6)
sos = make_sos(*PB[:2]); d = group_delay(sos, DATA[TRAIN_RECS[0]][0]); fx = apply(sos, s["noisy"])
pk = detect(fx, PB[2], d); fa = fx[d:]; n = min(len(fa), 12 * FS); t = np.arange(n) / FS; pk = pk[pk < n]
fig, ax = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
ax[0].plot(t, s["ref"][:n]); ax[0].set_title(f"Reference ECG (record {s['rec']})")
ax[1].plot(t, s["noisy"][:n], color="orange"); ax[1].set_title("Noisy ECG (muscle noise, -6 dB)")
ax[2].plot(t, fa[:n], color="green"); ax[2].plot(pk / FS, fa[pk], "kx", ms=9, label="detected R-peaks")
ax[2].set_title("Tuned causal filter + detector"); ax[2].set_xlabel("time (s)"); ax[2].legend()
plt.tight_layout(); plt.savefig("results/demo_signal.png", dpi=120); plt.show()

res.to_csv("results/test_results_all_scenarios.csv", index=False); ablation.round(4).to_csv("results/ablation.csv")
stab.round(4).to_csv("results/stability.csv", index=False); by.round(4).to_csv("results/f1_by_noise.csv")
with open("results/params_used.json", "w") as fh:
    json.dump(dict(mode=MODE, synthetic_data=SYNTHETIC, fs=FS, default=DEFAULT, tuned=dict(hp=float(PB[0]), lp=float(PB[1]), thr=float(PB[2])),
               pso=PSO, seeds=SEEDS, weights=W, bounds=dict(lo=LO.tolist(), hi=HI.tolist()), limits=dict(amp=AMP_MAX, st_mV=ST_MAX),
               train=TRAIN_RECS, test=TEST_RECS), fh, indent=2)
with zipfile.ZipFile("results.zip", "w") as z:
    for fn in sorted(glob.glob("results/*")): z.write(fn)
try:
    from google.colab import files; files.download("results.zip")
except Exception:
    print("Saved results.zip (not running in Colab, so no automatic download).")

## Write-up notes (for the jury)

- **Representation:** one PSO particle = `[high-pass Hz, low-pass Hz, detector threshold]`.
- **Operators:** inertia-weighted velocity update (own best + swarm best), velocity clamp, position clamp to bounds, inertia decay 0.9 → 0.4, particle 0 seeded with the baseline.
- **Why PSO (CI technique):** the fitness uses F1 and beat matching, so it is non-differentiable and has a jumpy surface; PSO needs no gradients and handles 3 continuous parameters with a tiny memory cost.
- **Constraints protected:** the fitness charges for QRS-amplitude and ST-level distortion (measured on the clean signal) and adds a large penalty when either hard limit is broken. A reward for SNR gain on baseline-wander scenarios stops PSO from simply switching the high-pass off. The weights in `W` are design choices: justify or tune them, and log every change in `attempts.md`.
- **Real-time honesty:** the filter and detector only use past samples (proved by the self-checks); the filter delay is measured and belongs in the 3 s budget.
- **Known limits:** the detector assumes a QRS is present (a flat or sine-only input can still produce false beats), so asystole and signal-quality checks live in Part 2. Only 60 s per record and one lead are used.
- **Not yet covered (Part 2):** per-beat features, fuzzy classifier, VF and asystole paths, alert timing, held-out VFDB/CUDB.